In [ ]:
%reload_ext autoreload
%autoreload 2

import os

import torch
import torch.nn as nn
import torchvision
from tqdm import tqdm

import pickle
from matplotlib import pyplot as plt

## 1. Two-Step对抗训练

- 在前两周的实验中，你已经实现了简单的单步对抗攻击（FGSM）和迭代对抗攻击（PGD）；

- 在本周的第一个实验中，请实现一个Two-Step对抗训练防御算法，并测试其在训练集、测试集上的预测表现，以及其对FGSM、PGD的防御效果；

- 具体实验步骤如下：

  1. 将代码文件（Python文件与Notebook文件）上传到服务器端根目录；

  2. 将样本数据（Week567_img_label.pkl）上传至服务器端data/目录下；

  3. 将之前训练的模型参数（lenet5.pt）上传至服务器端model/目录下；

  4. 依照提示，完成**Python文件**与**Notebook文件**中的TODO内容；

In [ ]:
from Week567_General_Code_Question import LeNet5, load_mnist, fgsm, pgd
from Week567_General_Code_Question import evaluate

In [ ]:
# Parameter
batch_size = 128
epsilon = 0.2
iter = 5
alpha = 0.07

In [ ]:
# Model
model = LeNet5()
model.load_state_dict(torch.load('model/lenet5.pt'))
model.eval()

# Data
criterion = nn.CrossEntropyLoss()
train_loader, test_loader = load_mnist(batch_size=batch_size)

### 生成对抗样本

In [ ]:
benign_imgs, benign_labels = [], []
fgsm_adv_imgs, pgd_adv_imgs, adv_labels = [], [], []

for img, label in tqdm(train_loader):
    benign_imgs.append(img)
    benign_labels.append(label)

    fgsm_adv_imgs.append(fgsm(img, epsilon, model, criterion, label))
    pgd_adv_imgs.append(pgd(img, epsilon, alpha, iter, model, criterion, label))
    adv_labels.append(label)

benign_imgs = torch.cat(benign_imgs, dim=0).detach()
benign_labels = torch.cat(benign_labels, dim=0).detach()
fgsm_adv_imgs = torch.cat(fgsm_adv_imgs, dim=0).detach()
pgd_adv_imgs = torch.cat(pgd_adv_imgs, dim=0).detach()
adv_labels = torch.cat(adv_labels, dim=0).detach()

In [ ]:
def build_mixed_trainloader(benign_imgs, benign_labels, adv_imgs, adv_labels, clean_ratio=1.0, adv_ratio=1.0, batch_size=128):
    def sample_by_ratio(imgs, labels, ratio):
        total = imgs.shape[0]
        count = int(total * ratio)
        full_repeat = count // total
        remain = count % total

        img_parts, label_parts = [], []
        if full_repeat > 0:
            img_parts.append(imgs.repeat((full_repeat, 1, 1, 1)))
            label_parts.append(labels.repeat(full_repeat))
        if remain > 0:
            img_parts.append(imgs[:remain])
            label_parts.append(labels[:remain])

        return torch.cat(img_parts, dim=0), torch.cat(label_parts, dim=0)

    clean_imgs, clean_labels = sample_by_ratio(benign_imgs, benign_labels, clean_ratio)
    mixed_adv_imgs, mixed_adv_labels = sample_by_ratio(adv_imgs, adv_labels, adv_ratio)

    mixed_imgs = torch.cat([clean_imgs, mixed_adv_imgs], dim=0)
    mixed_labels = torch.cat([clean_labels, mixed_adv_labels], dim=0)

    trainset = torch.utils.data.TensorDataset(mixed_imgs, mixed_labels)
    return torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)


clean_ratio = 1.0
adv_ratio = 1.0

fgsm_trainloader = build_mixed_trainloader(benign_imgs, benign_labels, fgsm_adv_imgs, adv_labels, clean_ratio, adv_ratio, batch_size)
pgd_trainloader = build_mixed_trainloader(benign_imgs, benign_labels, pgd_adv_imgs, adv_labels, clean_ratio, adv_ratio, batch_size)

### 实现Two-Step对抗训练
- 请在下面的block中实现基于FGSM/PGD的Two-Step对抗训练攻击
  - `adv_train_two_step(data_loader, epoch, lr, criterion)`
- 算法流程
  - 先将正常样本和预生成的对抗样本按设定比例混合为一个 `TensorDataset`
  - 再从 `dataloader` 中随机取出混合后的样本，直接计算一个总的分类损失并反向传播更新模型

In [ ]:
def adv_train_two_step(data_loader, epoch, lr, criterion):
    model = LeNet5()
    model.load_state_dict(torch.load('model/lenet5.pt'))
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for e in range(epoch):
        t = tqdm(data_loader)
        for img, label in t:
            # TODO: 前向传播并在混合batch上计算分类损失
            pred = None
            loss = None

            # TODO: 反向更新模型参数

            t.set_postfix(epoch=e, loss=loss.item())

    return model

- 使用fgsm进行对抗训练

In [ ]:
lr = 0.01
epoch = 10

cnn_fgsm_two_step = adv_train_two_step(fgsm_trainloader, epoch, lr, criterion)
torch.save(cnn_fgsm_two_step.state_dict(), 'model/cnn_fgsm_two_step.pt')

- 使用pgd进行对抗训练

In [ ]:
lr = 0.01
epoch = 10

cnn_pgd_two_step = adv_train_two_step(pgd_trainloader, epoch, lr, criterion)
torch.save(cnn_pgd_two_step.state_dict(), 'model/cnn_pgd_two_step.pt')
# 训练后保存好模型文件，以便检查时快速测试结果

### 评测模型性能
- 请在Python文件Week567_General_Code_Question.py中补全函数如下：
  - 在`evaluate_dataloader(dataloader, model)`函数实现模型测试过程

In [ ]:
from Week567_General_Code_Question import evaluate_dataloader

- 测试基于FGSM执行Two-Step对抗训练的CNN的预测质量

In [ ]:
evaluate_dataloader(test_loader, cnn_fgsm_two_step)

- 测试基于PGD执行Two-Step对抗训练的CNN的预测质量

In [ ]:
evaluate_dataloader(test_loader, cnn_pgd_two_step)

### 评测防御效果

In [ ]:
with open('data/Week567_img_label.pkl', 'rb') as f:
    data = pickle.load(f)
    imgs, labels = data['img'], data['label']

- 评测原始模型针对FGSM/PGD攻击的效果（作为参照）

In [ ]:
print("For Original Model.\n")
print("Against FGSM:")
epsilon = 0.2

adv_xs = fgsm(imgs, epsilon, model, criterion, labels)
pred_label = evaluate(adv_xs, labels, model)


print("Against PGD:")
alpha = 0.07
iter = 5

adv_xs = pgd(imgs, epsilon, alpha, iter, model, criterion, labels)

pred_label = evaluate(adv_xs, labels, model)

- 评测基于FGSM执行Two-Step对抗训练的模型针对FGSM/PGD攻击的防御效果

In [ ]:
print("For FGSM Two-Step.\n")
print("Against FGSM:")
epsilon = 0.2

adv_xs = fgsm(imgs, epsilon, cnn_fgsm_two_step, criterion, labels)
pred_label = evaluate(adv_xs, labels, cnn_fgsm_two_step)


print("Against PGD:")
alpha = 0.07
iter = 5

adv_xs = pgd(imgs, epsilon, alpha, iter, cnn_fgsm_two_step, criterion, labels)

pred_label = evaluate(adv_xs, labels, cnn_fgsm_two_step)

- 评测基于PGD执行Two-Step对抗训练的模型针对FGSM/PGD攻击的防御效果

In [ ]:
print("For PGD Two-Step.\n")
print("Against FGSM:")
epsilon = 0.2

adv_xs = fgsm(imgs, epsilon, cnn_pgd_two_step, criterion, labels)
pred_label = evaluate(adv_xs, labels, cnn_pgd_two_step)


print("Against PGD:")
alpha = 0.07
iter = 5

adv_xs = pgd(imgs, epsilon, alpha, iter, cnn_pgd_two_step, criterion, labels)

pred_label = evaluate(adv_xs, labels, cnn_pgd_two_step)